# LENS / DeiT-Tiny: Joint LearnedNorm + MLPSoftmax (Base-2)

This notebook starts from the official pretrained **timm DeiT-Tiny**
(`deit_tiny_patch16_224.fb_in1k`) and converts it directly to the current
full LENS design.

The conversion is kept consistent with the CIFAR-10 / CIFAR-100 experiments:

1. replace every LayerNorm with LearnedNorm;
2. replace every analytical attention Softmax with row-wise MLPSoftmax;
3. **unfreeze the whole model immediately**;
4. jointly fine-tune all parameters end-to-end with one learning rate.

There is **no three-stage training** and no frozen-backbone warm-up.

LearnedNorm uses

\[
[a_t,b_t] = f_\theta(x_t), \qquad y_t = a_t x_t + b_t.
\]

The slope `a` keeps the `[0.05, 12]` clamp. The dynamic bias `b`
is **not clamped**.

MLPSoftmax uses the complete score row:

\[
s_{q,:} \in \mathbb{R}^N
\rightarrow \mathrm{MLP}
\rightarrow L_q,
\qquad
A_{qi} = 2^{s_{qi} - L_q}.
\]

Runtime additions:

- every completed epoch is appended to `training_log.txt`;
- `last.pt` is saved every epoch and training auto-resumes from it;
- checkpoints are loaded through CPU first, so a run trained on `cuda:0`
  can be resumed/evaluated on `cuda:1` or another visible device;
- checkpoint metadata is checked to prevent loading the wrong LENS variant.


In [1]:

# If timm is not installed in this environment, run this cell once:
# %pip install -U timm


In [2]:
# ============================================================
# Imports and configuration
# ============================================================

import json
import math
import os
import random
import time
from contextlib import nullcontext
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode

import timm
from timm.layers import LayerNorm as TimmLayerNorm
from timm.layers import maybe_add_mask, resolve_self_attn_mask

# ------------------------------------------------------------
# Reproducibility / runtime
# ------------------------------------------------------------
SEED = 2333
NUM_WORKERS = 8
BATCH_SIZE = 256
PIN_MEMORY = True

torch.set_float32_matmul_precision("high")
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

USE_AMP = True
AMP_DTYPE = torch.bfloat16

# None -> current/default visible CUDA device.
# 0/1/... -> explicit logical CUDA index.
# If CUDA_VISIBLE_DEVICES is set externally, leave this as None.
CUDA_DEVICE_INDEX = 0

# ------------------------------------------------------------
# ImageNet-1K
# ------------------------------------------------------------
IMAGENET_ROOT = Path("data/ImageNet")

MAX_TRAIN_SAMPLES = None
MAX_VAL_SAMPLES = None

# ------------------------------------------------------------
# timm DeiT-Tiny pretrained model
# ------------------------------------------------------------
MODEL_NAME = "deit_tiny_patch16_224.fb_in1k"
NUM_CLASSES = 1000

LEARNED_NORM_HIDDEN = 32
SOFTMAX_HIDDEN = 32

# Match the current CIFAR LearnedNorm behavior:
# keep only the slope clamp; do not clamp b/beta.
LEARNED_NORM_A_MIN = 0.05
LEARNED_NORM_A_MAX = 12.0

SOFTMAX_BASE = "2"
RUN_VARIANT = "base_2"

WEIGHT_DECAY = 0.05
GRAD_CLIP_NORM = 1.0

# ------------------------------------------------------------
# Direct joint full-model training
# ------------------------------------------------------------
JOINT_EPOCHS = 50
JOINT_LR = 7.5e-5

JOINT_OUT_DIR = Path("./runs/lens_deit_tiny_joint_mlp_base2")
JOINT_OUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_CKPT = JOINT_OUT_DIR / "best.pt"
LAST_CKPT = JOINT_OUT_DIR / "last.pt"
LOG_FILE = JOINT_OUT_DIR / "training_log.txt"
HISTORY_FILE = JOINT_OUT_DIR / "history.json"
BASELINE_METRICS_FILE = JOINT_OUT_DIR / "baseline_metrics.json"

AUTO_RESUME = True


/home/yuhao/anaconda3/envs/brevitas/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ============================================================
# Reproducibility and selected device
# ============================================================

def seed_cpu_side(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_cpu_side(SEED)

if torch.cuda.is_available():
    if CUDA_DEVICE_INDEX is None:
        selected_cuda_index = torch.cuda.current_device()
    else:
        selected_cuda_index = int(CUDA_DEVICE_INDEX)
        if selected_cuda_index < 0 or selected_cuda_index >= torch.cuda.device_count():
            raise ValueError(
                f"CUDA_DEVICE_INDEX={selected_cuda_index} is invalid; "
                f"{torch.cuda.device_count()} CUDA device(s) are visible."
            )

    torch.cuda.set_device(selected_cuda_index)
    device = torch.device(f"cuda:{selected_cuda_index}")

    # Seed only the selected logical CUDA device.
    torch.cuda.manual_seed(SEED)

    print("Device:", device)
    print("GPU:", torch.cuda.get_device_name(selected_cuda_index))
    print("Visible CUDA devices:", torch.cuda.device_count())
else:
    device = torch.device("cpu")
    print("Device:", device)


Device: cuda:0
GPU: NVIDIA A40
Visible CUDA devices: 2



## 1. Load pretrained DeiT-Tiny

`timm.create_model(..., pretrained=True)` downloads the pretrained ImageNet-1K weights automatically on the first run and then uses the local cache on later runs.

The standard model is evaluated **before any operator is replaced**, giving a local baseline with exactly the same ImageNet validation loader used for the converted model.


In [4]:

# ============================================================
# Download / load pretrained DeiT-Tiny
# ============================================================

model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=NUM_CLASSES,
)

# Resolve preprocessing directly from the pretrained model metadata.
data_config = timm.data.resolve_model_data_config(model)
print("Resolved data config:", data_config)

print("\nModel summary:")
print("  blocks:", len(model.blocks))
print("  embed dim:", model.num_features)
print("  tokens from pos_embed:", model.pos_embed.shape[1])
print("  heads in block 0:", model.blocks[0].attn.num_heads)
print("  head dim in block 0:", model.blocks[0].attn.head_dim)
print("  parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

# Structural checks: fail early if a future timm version changes the model API.
assert hasattr(model, "blocks") and len(model.blocks) > 0
for i, block in enumerate(model.blocks):
    assert hasattr(block, "attn"), f"block {i} has no attn module"
    for attr in ["qkv", "q_norm", "k_norm", "attn_drop", "proj", "proj_drop", "num_heads", "head_dim", "scale"]:
        assert hasattr(block.attn, attr), f"block {i}.attn is missing '{attr}'"

Resolved data config: {'input_size': (3, 224, 224), 'interpolation': 'bicubic', 'mean': (0.485, 0.456, 0.406), 'std': (0.229, 0.224, 0.225), 'crop_pct': 0.9, 'crop_mode': 'center'}

Model summary:
  blocks: 12
  embed dim: 192
  tokens from pos_embed: 197
  heads in block 0: 3
  head dim in block 0: 64
  parameters: 5,717,416


## 2. ImageNet-1K loaders

The validation transform is taken from the timm pretrained configuration.
The training transform uses the same image size / normalization with standard
random resized crop and horizontal flip.

This notebook expects:

- `data/ImageNet/ILSVRC2012_img_train/<class>/*.JPEG`
- `data/ImageNet/ILSVRC2012_img_val/<class>/*.JPEG`


In [5]:
# ============================================================
# ImageNet-1K data
# ============================================================

train_dir = IMAGENET_ROOT / "ILSVRC2012_img_train"
val_dir = IMAGENET_ROOT / "ILSVRC2012_img_val"

if not train_dir.exists() or not val_dir.exists():
    raise FileNotFoundError(
        "IMAGENET_ROOT must contain train/ and val/ class subdirectories. "
        f"Current path: {IMAGENET_ROOT.resolve()}"
    )

input_size = data_config["input_size"][-1]
mean = data_config["mean"]
std = data_config["std"]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        input_size,
        interpolation=InterpolationMode.BICUBIC,
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

eval_transform = timm.data.create_transform(
    **data_config,
    is_training=False,
)

train_set = datasets.ImageFolder(train_dir, transform=train_transform)
val_set = datasets.ImageFolder(val_dir, transform=eval_transform)

if MAX_TRAIN_SAMPLES is not None:
    train_set = Subset(train_set, range(min(MAX_TRAIN_SAMPLES, len(train_set))))
if MAX_VAL_SAMPLES is not None:
    val_set = Subset(val_set, range(min(MAX_VAL_SAMPLES, len(val_set))))

loader_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = True

train_loader = DataLoader(
    train_set,
    shuffle=True,
    drop_last=True,
    **loader_kwargs,
)

val_loader = DataLoader(
    val_set,
    shuffle=False,
    drop_last=False,
    **loader_kwargs,
)

print("Train images:", len(train_set))
print("Val images:", len(val_set))
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))


Train images: 1281154
Val images: 50000
Train batches: 5004
Val batches: 196


## 3. Learned replacements

### LearnedNorm

\[
[a_t,b_t] = f_\theta(x_t), \qquad y_t=a_t x_t+b_t.
\]

`a_t` is clamped to `[0.05, 12]`. `b_t` is **unclamped**.

### MLPSoftmax (Base-2)

For every complete attention-score row:

\[
s_{q,:}\in\mathbb{R}^{197}
\rightarrow 197\rightarrow32\rightarrow1
\rightarrow L_q.
\]

The MLP is evaluated in parallel across batch/head/query rows. There is
no recurrent scan.

\[
A_{qi} = 2^{s_{qi} - L_q}.
\]

The exponent input is clamped to `[-12, 0]`, matching the CIFAR version.


In [6]:
# ============================================================
# Learned LayerNorm replacement
# ============================================================

class LearnedLayerNormReplacement(nn.Module):
    """
    Dynamic affine LayerNorm replacement:
        [a_t, b_t] = MLP(x_t)
        y_t = a_t * x_t + b_t

    a_t is clamped; b_t is intentionally not clamped.
    """

    def __init__(self, dim, hidden_dim=32, name="ln"):
        super().__init__()
        self.dim = dim
        self.name = name

        self.predictor_0 = nn.Linear(dim, hidden_dim, bias=True)
        self.predictor_relu = nn.ReLU()
        self.predictor_1 = nn.Linear(hidden_dim, 2, bias=True)

        nn.init.kaiming_uniform_(
            self.predictor_0.weight,
            nonlinearity="relu",
        )
        nn.init.zeros_(self.predictor_0.bias)

        nn.init.normal_(
            self.predictor_1.weight,
            mean=0.0,
            std=1e-3,
        )

        with torch.no_grad():
            self.predictor_1.bias[0] = 1.0
            self.predictor_1.bias[1] = 0.0

    def forward(self, x):
        state = self.predictor_0(x)
        state = self.predictor_relu(state)
        state = self.predictor_1(state)

        raw_a = state[..., 0:1]
        b = state[..., 1:2]  # no beta/b clamp

        a = torch.clamp(
            raw_a,
            min=LEARNED_NORM_A_MIN,
            max=LEARNED_NORM_A_MAX,
        )

        return a * x + b


# ============================================================
# Row-wise MLPSoftmax replacement, multi-head version
# ============================================================

class MLPSoftmaxReplacement(nn.Module):
    """
    scores: [B, H, N_query, N_key]

    Every complete score row is mapped by:
        N_key -> hidden_dim -> 1
    to one scalar normalization state L_q.
    """

    def __init__(self, num_tokens, hidden_dim=32):
        super().__init__()
        self.num_tokens = int(num_tokens)

        self.normalizer_mlp = nn.Sequential(
            nn.Linear(self.num_tokens, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

        first_layer = self.normalizer_mlp[0]
        final_layer = self.normalizer_mlp[-1]

        nn.init.trunc_normal_(first_layer.weight, std=0.02)
        nn.init.zeros_(first_layer.bias)

        nn.init.normal_(final_layer.weight, mean=0.0, std=1e-3)
        nn.init.constant_(final_layer.bias, math.log2(self.num_tokens))

    def forward(self, scores):
        if scores.ndim != 4:
            raise ValueError(
                f"Expected scores [B,H,Nq,Nk], got {tuple(scores.shape)}"
            )

        B, H, Nq, Nk = scores.shape

        if Nk != self.num_tokens:
            raise ValueError(
                f"MLPSoftmaxReplacement expects N_key={self.num_tokens}, "
                f"but received N_key={Nk}."
            )

        rows = scores.reshape(B * H * Nq, Nk)
        L = self.normalizer_mlp(rows)
        L = L.reshape(B, H, Nq, 1)

        exponent = torch.clamp(
            scores - L,
            min=-21.0,
            max=-7.0,
        )

        return torch.exp2(exponent)


## 4. timm attention wrapper with MLPSoftmax (Base-2)

The wrapper explicitly reconstructs Q/K/V attention so the analytical
Softmax can be replaced. It reuses the pretrained QKV, projection,
normalization, dropout, and optional gate modules by reference.

Each attention block gets its own `197 -> 32 -> 1` MLPSoftmax predictor,
shared across that block's heads and query rows.


In [7]:
# ============================================================
# Wrapper around timm Attention that preserves pretrained weights
# ============================================================

class TimmAttentionWithMLPSoftmax(nn.Module):
    """
    Drop-in wrapper for timm ViT/DeiT Attention.
    Only analytical Softmax is replaced.
    """

    def __init__(
        self,
        original_attn,
        num_tokens,
        hidden_dim=32,
    ):
        super().__init__()

        self.num_heads = original_attn.num_heads
        self.head_dim = original_attn.head_dim
        self.attn_dim = getattr(
            original_attn,
            "attn_dim",
            self.num_heads * self.head_dim,
        )
        self.scale = original_attn.scale

        self.qkv = original_attn.qkv
        self.q_norm = original_attn.q_norm
        self.k_norm = original_attn.k_norm
        self.attn_drop = original_attn.attn_drop
        self.norm = getattr(original_attn, "norm", nn.Identity())
        self.gate = getattr(original_attn, "gate", None)
        self.proj = original_attn.proj
        self.proj_drop = original_attn.proj_drop

        self.learned_softmax = MLPSoftmaxReplacement(
            num_tokens=num_tokens,
            hidden_dim=hidden_dim,
        )

    def forward(
        self,
        x,
        attn_mask=None,
        is_causal=False,
    ):
        B, N, C = x.shape

        gate = self.gate(x).sigmoid() if self.gate is not None else None

        qkv = self.qkv(x).reshape(
            B,
            N,
            3,
            self.num_heads,
            self.head_dim,
        ).permute(2, 0, 3, 1, 4)

        q, k, v = qkv.unbind(0)
        q = self.q_norm(q)
        k = self.k_norm(k)

        q = q * self.scale
        scores = q @ k.transpose(-2, -1)

        attn_bias = resolve_self_attn_mask(
            N,
            scores,
            attn_mask,
            is_causal,
        )
        scores = maybe_add_mask(scores, attn_bias)

        attn = self.learned_softmax(scores)
        attn = self.attn_drop(attn)

        x = attn @ v
        x = x.transpose(1, 2).reshape(B, N, self.attn_dim)

        x = self.norm(x)
        if gate is not None:
            x = x * gate

        x = self.proj(x)
        x = self.proj_drop(x)
        return x


## 5. Automatic replacement helpers

LayerNorm replacement walks the whole model recursively. Attention replacement
wraps each DeiT attention block while preserving its pretrained QKV/projection
parameters. After both replacements, the **entire model is trainable**.


In [8]:
# ============================================================
# Automatic replacement helpers
# ============================================================

def _is_layernorm(module):
    return isinstance(module, (nn.LayerNorm, TimmLayerNorm))


def replace_all_layernorms(
    module,
    hidden_dim=32,
    prefix="",
    replaced=None,
):
    if replaced is None:
        replaced = []

    for child_name, child in list(module.named_children()):
        full_name = f"{prefix}.{child_name}" if prefix else child_name

        if isinstance(child, LearnedLayerNormReplacement):
            continue

        if _is_layernorm(child):
            normalized_shape = child.normalized_shape

            if isinstance(normalized_shape, int):
                dim = normalized_shape
            else:
                if len(normalized_shape) != 1:
                    raise ValueError(
                        f"Only 1-D LayerNorm is supported, "
                        f"got {normalized_shape} at {full_name}"
                    )
                dim = normalized_shape[0]

            setattr(
                module,
                child_name,
                LearnedLayerNormReplacement(
                    dim=dim,
                    hidden_dim=hidden_dim,
                    name=full_name,
                ),
            )
            replaced.append(full_name)
        else:
            replace_all_layernorms(
                child,
                hidden_dim=hidden_dim,
                prefix=full_name,
                replaced=replaced,
            )

    return replaced


def replace_all_attention_softmax(
    model,
    hidden_dim=32,
):
    num_tokens = int(model.pos_embed.shape[1])
    replaced = []

    for block_id, block in enumerate(model.blocks):
        if isinstance(block.attn, TimmAttentionWithMLPSoftmax):
            continue

        block.attn = TimmAttentionWithMLPSoftmax(
            original_attn=block.attn,
            num_tokens=num_tokens,
            hidden_dim=hidden_dim,
        )
        replaced.append(f"blocks.{block_id}.attn")

    return replaced


def unfreeze_all(model):
    for p in model.parameters():
        p.requires_grad = True


def parameter_counts(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )
    return trainable, total


def print_trainable_parameters(model, title=""):
    trainable, total = parameter_counts(model)

    if title:
        print(title)

    print(
        f"Trainable: {trainable:,} / {total:,} "
        f"({100.0 * trainable / total:.4f}%)"
    )


## 6. Training / evaluation / logging / resume helpers

`last.pt` is written after every completed epoch. If the run is interrupted,
rerunning the notebook with `AUTO_RESUME=True` resumes from the next epoch.

Checkpoints are always deserialized to **CPU first** and then moved to the
currently selected CUDA device. Optimizer-state tensors are also explicitly
moved, so switching from one CUDA index to another is supported.

`training_log.txt` receives one line after every completed epoch.


In [9]:
# ============================================================
# Training / evaluation / checkpoint / logging helpers
# ============================================================

criterion = nn.CrossEntropyLoss()


def autocast_context(device):
    if USE_AMP and device.type == "cuda":
        return torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        )
    return nullcontext()


def sync_device(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def format_seconds(seconds):
    seconds = float(seconds)

    if seconds < 60:
        return f"{seconds:.1f}s"

    minutes, sec = divmod(seconds, 60)

    if minutes < 60:
        return f"{int(minutes)}m {sec:.1f}s"

    hours, minutes = divmod(minutes, 60)
    return f"{int(hours)}h {int(minutes)}m {sec:.1f}s"


def append_log(path, message):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().isoformat(timespec="seconds")

    with path.open("a", encoding="utf-8") as f:
        f.write(f"[{timestamp}] {message}\n")
        f.flush()


def portable_torch_load(path):
    """
    Always deserialize tensors onto CPU first.
    This removes dependence on the CUDA index used when saving.
    """
    path = Path(path)

    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location="cpu",
        )


def move_nested_to_device(obj, device):
    if torch.is_tensor(obj):
        return obj.to(device)

    if isinstance(obj, dict):
        return {
            k: move_nested_to_device(v, device)
            for k, v in obj.items()
        }

    if isinstance(obj, list):
        return [
            move_nested_to_device(v, device)
            for v in obj
        ]

    if isinstance(obj, tuple):
        return tuple(
            move_nested_to_device(v, device)
            for v in obj
        )

    return obj


def optimizer_to(optimizer, device):
    for param_state in optimizer.state.values():
        for key, value in list(param_state.items()):
            param_state[key] = move_nested_to_device(
                value,
                device,
            )


def atomic_torch_save(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    tmp_path = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp_path)
    os.replace(tmp_path, path)


def expected_checkpoint_config(model):
    return {
        "model_name": MODEL_NAME,
        "num_classes": NUM_CLASSES,
        "num_tokens": int(model.pos_embed.shape[1]),
        "learned_norm_hidden": LEARNED_NORM_HIDDEN,
        "softmax_hidden": SOFTMAX_HIDDEN,
        "softmax_base": SOFTMAX_BASE,
        "run_variant": RUN_VARIANT,
        "joint_epochs": JOINT_EPOCHS,
    }


def validate_checkpoint_config(ckpt, model):
    expected = expected_checkpoint_config(model)
    found = ckpt.get("config", None)

    if found is None:
        raise RuntimeError(
            "Checkpoint has no LENS config metadata. "
            "Automatic loading is refused because the variant "
            "cannot be verified."
        )

    mismatches = []

    for key, expected_value in expected.items():
        found_value = found.get(key, None)

        if found_value != expected_value:
            mismatches.append(
                f"{key}: checkpoint={found_value!r}, "
                f"current={expected_value!r}"
            )

    if mismatches:
        raise RuntimeError(
            "Checkpoint configuration mismatch:\n  "
            + "\n  ".join(mismatches)
            + "\nUse the matching notebook/output directory, "
              "or start a fresh run."
        )


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()

    total = 0
    correct1 = 0
    correct5 = 0
    loss_sum = 0.0

    sync_device(device)
    start_time = time.perf_counter()

    for images, labels in loader:
        images = images.to(
            device,
            non_blocking=True,
        )
        labels = labels.to(
            device,
            non_blocking=True,
        )

        with autocast_context(device):
            logits = model(images)
            loss = criterion(logits, labels)

        batch_size = labels.size(0)
        total += batch_size
        loss_sum += loss.item() * batch_size

        top5 = logits.topk(5, dim=1).indices
        correct1 += (
            top5[:, 0] == labels
        ).sum().item()
        correct5 += (
            top5.eq(labels[:, None])
            .any(dim=1)
            .sum()
            .item()
        )

    sync_device(device)
    elapsed = time.perf_counter() - start_time

    return {
        "loss": loss_sum / total,
        "top1": correct1 / total,
        "top5": correct5 / total,
        "time_sec": elapsed,
    }


def print_eval(
    model,
    loader,
    device,
    title,
    split_name="Validation",
):
    metrics = evaluate(
        model,
        loader,
        device,
    )

    print(f"\n[{title}]")
    print(
        f"{split_name}: loss={metrics['loss']:.4f}, "
        f"top1={metrics['top1']:.4f}, "
        f"top5={metrics['top5']:.4f}, "
        f"time={format_seconds(metrics['time_sec'])}"
    )

    return metrics


def load_model_checkpoint(
    model,
    path,
    device,
    validate_config=True,
    strict=True,
):
    path = Path(path)
    ckpt = portable_torch_load(path)

    if validate_config:
        validate_checkpoint_config(
            ckpt,
            model,
        )

    model.load_state_dict(
        ckpt["model"],
        strict=strict,
    )
    model.to(device)

    val = ckpt.get("val", {})

    print(
        f"Loaded checkpoint: {path} | "
        f"epoch={ckpt.get('epoch', '?')} | "
        f"val_top1={val.get('top1', '?')} | "
        f"current device={device}"
    )

    return ckpt


def train_joint_model(
    model,
    train_loader,
    val_loader,
    device,
    epochs,
    lr,
    weight_decay,
    out_dir,
    grad_clip_norm=1.0,
    auto_resume=True,
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    last_path = out_dir / "last.pt"
    best_path = out_dir / "best.pt"
    log_path = out_dir / "training_log.txt"
    history_path = out_dir / "history.json"

    trainable_params = [
        p for p in model.parameters()
        if p.requires_grad
    ]

    if not trainable_params:
        raise RuntimeError(
            "Joint training: no trainable parameters selected."
        )

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=lr,
        weight_decay=weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs,
    )

    trainable_count = sum(
        p.numel() for p in trainable_params
    )
    total_count = sum(
        p.numel() for p in model.parameters()
    )

    start_epoch = 1
    best_val_top1 = -float("inf")
    history = []

    if auto_resume and last_path.exists():
        ckpt = portable_torch_load(last_path)
        validate_checkpoint_config(
            ckpt,
            model,
        )

        model.load_state_dict(
            ckpt["model"],
            strict=True,
        )
        model.to(device)

        optimizer.load_state_dict(
            ckpt["optimizer"],
        )
        optimizer_to(
            optimizer,
            device,
        )

        scheduler.load_state_dict(
            ckpt["scheduler"],
        )

        start_epoch = int(ckpt["epoch"]) + 1
        best_val_top1 = float(
            ckpt.get(
                "best_val_top1",
                -float("inf"),
            )
        )
        history = list(
            ckpt.get("history", [])
        )

        resume_message = (
            f"RESUME from {last_path} | "
            f"last completed epoch={start_epoch - 1} | "
            f"next epoch={start_epoch} | "
            f"device={device}"
        )

        print("\n" + resume_message)
        append_log(
            log_path,
            resume_message,
        )

    else:
        with log_path.open(
            "w",
            encoding="utf-8",
        ) as f:
            f.write("")

        with history_path.open(
            "w",
            encoding="utf-8",
        ) as f:
            json.dump([], f)

        append_log(
            log_path,
            (
                f"NEW RUN | variant={RUN_VARIANT} | "
                f"device={device} | "
                f"lr={lr:.3e} | epochs={epochs}"
            ),
        )

    print(
        f"\nJoint full-model training: "
        f"trainable parameters {trainable_count:,} / {total_count:,} "
        f"({100.0 * trainable_count / total_count:.4f}%)"
    )
    print(
        f"Joint full-model training: "
        f"lr={optimizer.param_groups[0]['lr']:.3e}, "
        f"target epochs={epochs}, "
        f"start epoch={start_epoch}"
    )
    print(
        "Text log:",
        log_path.resolve(),
    )

    append_log(
        log_path,
        (
            f"Trainable={trainable_count:,}/{total_count:,} "
            f"({100.0 * trainable_count / total_count:.4f}%)"
        ),
    )

    if start_epoch > epochs:
        message = (
            f"Training already completed through epoch "
            f"{start_epoch - 1}; target is {epochs}. "
            f"No additional epochs are needed."
        )
        print(message)
        append_log(
            log_path,
            message,
        )
        return {
            "best_val_top1": best_val_top1,
            "history": history,
            "start_epoch": start_epoch,
        }

    for epoch in range(
        start_epoch,
        epochs + 1,
    ):
        model.train()

        total = 0
        correct1 = 0
        loss_sum = 0.0
        max_grad_norm = 0.0

        sync_device(device)
        train_start = time.perf_counter()

        for images, labels in train_loader:
            images = images.to(
                device,
                non_blocking=True,
            )
            labels = labels.to(
                device,
                non_blocking=True,
            )

            optimizer.zero_grad(
                set_to_none=True,
            )

            with autocast_context(device):
                logits = model(images)
                loss = criterion(
                    logits,
                    labels,
                )

            loss.backward()

            if grad_clip_norm is not None:
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    trainable_params,
                    max_norm=grad_clip_norm,
                )

                try:
                    grad_norm_value = float(
                        grad_norm.detach().cpu()
                    )

                    if math.isfinite(grad_norm_value):
                        max_grad_norm = max(
                            max_grad_norm,
                            grad_norm_value,
                        )
                except Exception:
                    pass

            optimizer.step()

            batch_size = labels.size(0)
            total += batch_size
            loss_sum += (
                loss.item() * batch_size
            )
            correct1 += (
                logits.argmax(dim=1) == labels
            ).sum().item()

        sync_device(device)
        train_time = (
            time.perf_counter()
            - train_start
        )

        scheduler.step()

        train_loss = loss_sum / total
        train_top1 = correct1 / total

        val_metrics = evaluate(
            model,
            val_loader,
            device,
        )

        current_lr = (
            scheduler.get_last_lr()[0]
        )

        record = {
            "epoch": epoch,
            "lr": current_lr,
            "train_loss": train_loss,
            "train_top1": train_top1,
            "val_loss": val_metrics["loss"],
            "val_top1": val_metrics["top1"],
            "val_top5": val_metrics["top5"],
            "train_time_sec": train_time,
            "val_time_sec": val_metrics["time_sec"],
            "max_finite_grad_norm_before_clipping": max_grad_norm,
        }
        history.append(record)

        line = (
            f"Epoch {epoch:03d}/{epochs} | "
            f"lr={current_lr:.3e} | "
            f"train loss={train_loss:.4f} "
            f"top1={train_top1:.4f} | "
            f"val loss={val_metrics['loss']:.4f} "
            f"top1={val_metrics['top1']:.4f} "
            f"top5={val_metrics['top5']:.4f} | "
            f"train time={format_seconds(train_time)} | "
            f"val time={format_seconds(val_metrics['time_sec'])} | "
            f"max grad norm={max_grad_norm:.3e}"
        )

        print(line)
        append_log(
            log_path,
            line,
        )

        is_new_best = (
            val_metrics["top1"]
            > best_val_top1
        )

        if is_new_best:
            best_val_top1 = (
                val_metrics["top1"]
            )

        checkpoint_common = {
            "stage": "Joint full-model LENS",
            "epoch": epoch,
            "model": model.state_dict(),
            "val": val_metrics,
            "best_val_top1": best_val_top1,
            "model_name": MODEL_NAME,
            "config": expected_checkpoint_config(model),
            "history": history,
            "saved_from_device": str(device),
        }

        if is_new_best:
            atomic_torch_save(
                checkpoint_common,
                best_path,
            )

        atomic_torch_save(
            {
                **checkpoint_common,
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
            },
            last_path,
        )

        with history_path.open(
            "w",
            encoding="utf-8",
        ) as f:
            json.dump(
                history,
                f,
                indent=2,
            )

    done_message = (
        f"DONE | best validation top1={best_val_top1:.4f} | "
        f"last completed epoch={epochs}"
    )

    print(done_message)
    append_log(
        log_path,
        done_message,
    )

    return {
        "best_val_top1": best_val_top1,
        "history": history,
        "start_epoch": start_epoch,
    }


## 7. Standard pretrained baseline

The untouched timm checkpoint is evaluated before conversion.

Baseline metrics are cached in the run directory so an interrupted run does
not need to repeat the standard ImageNet validation every time the notebook
is restarted.


In [10]:
# ============================================================
# Standard pretrained baseline
# ============================================================

model = model.to(device)

if BASELINE_METRICS_FILE.exists():
    with BASELINE_METRICS_FILE.open(
        "r",
        encoding="utf-8",
    ) as f:
        standard_metrics = json.load(f)

    print(
        "Loaded cached standard baseline metrics:",
        BASELINE_METRICS_FILE,
    )
    print(
        f"Validation: loss={standard_metrics['loss']:.4f}, "
        f"top1={standard_metrics['top1']:.4f}, "
        f"top5={standard_metrics['top5']:.4f}"
    )
else:
    standard_metrics = print_eval(
        model,
        val_loader,
        device,
        title="Pretrained standard DeiT-Tiny",
        split_name="Validation",
    )

    with BASELINE_METRICS_FILE.open(
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            standard_metrics,
            f,
            indent=2,
        )



[Pretrained standard DeiT-Tiny]
Validation: loss=1.2179, top1=0.7221, top5=0.9110, time=34.9s


## 8. Direct joint full-LENS fine-tuning (Base-2)

LayerNorm and Softmax are replaced **simultaneously**, the whole converted
DeiT-Tiny is unfrozen, and all parameters are jointly fine-tuned with one
learning rate.

If `last.pt` exists and `AUTO_RESUME=True`, model, optimizer, scheduler,
best validation score, and completed history are restored automatically.


In [11]:
# ============================================================
# Install full LENS and jointly fine-tune the entire model
# ============================================================

replaced_norms = replace_all_layernorms(
    model,
    hidden_dim=LEARNED_NORM_HIDDEN,
)

replaced_attn = replace_all_attention_softmax(
    model,
    hidden_dim=SOFTMAX_HIDDEN,
)

model = model.to(device)
unfreeze_all(model)

print(
    f"Replaced {len(replaced_norms)} LayerNorm modules."
)
for name in replaced_norms:
    print("  LN:", name)

print(
    f"Replaced Softmax in {len(replaced_attn)} attention modules."
)
for name in replaced_attn:
    print("  Attn:", name)

print("\n=== Joint full-LENS training ===")
print_trainable_parameters(model)

training_result = train_joint_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    epochs=JOINT_EPOCHS,
    lr=JOINT_LR,
    weight_decay=WEIGHT_DECAY,
    out_dir=JOINT_OUT_DIR,
    grad_clip_norm=GRAD_CLIP_NORM,
    auto_resume=AUTO_RESUME,
)

if not BEST_CKPT.exists():
    raise FileNotFoundError(
        f"Best checkpoint was not created: {BEST_CKPT}"
    )

load_model_checkpoint(
    model,
    BEST_CKPT,
    device,
    validate_config=True,
    strict=True,
)

final_metrics = print_eval(
    model,
    val_loader,
    device,
    title="Best full-LENS DeiT-Tiny",
    split_name="Validation",
)

print("\n=== Summary ===")
print(
    f"Standard pretrained top1: "
    f"{standard_metrics['top1']:.4f} | "
    f"top5: {standard_metrics['top5']:.4f}"
)
print(
    f"Full LENS top1:           "
    f"{final_metrics['top1']:.4f} | "
    f"top5: {final_metrics['top5']:.4f}"
)
print(
    f"Top-1 delta vs standard:  "
    f"{100.0 * (final_metrics['top1'] - standard_metrics['top1']):+.3f} pp"
)
print("Best checkpoint:", BEST_CKPT.resolve())
print("Resume checkpoint:", LAST_CKPT.resolve())
print("Epoch text log:", LOG_FILE.resolve())


Replaced 25 LayerNorm modules.
  LN: blocks.0.norm1
  LN: blocks.0.norm2
  LN: blocks.1.norm1
  LN: blocks.1.norm2
  LN: blocks.2.norm1
  LN: blocks.2.norm2
  LN: blocks.3.norm1
  LN: blocks.3.norm2
  LN: blocks.4.norm1
  LN: blocks.4.norm2
  LN: blocks.5.norm1
  LN: blocks.5.norm2
  LN: blocks.6.norm1
  LN: blocks.6.norm2
  LN: blocks.7.norm1
  LN: blocks.7.norm2
  LN: blocks.8.norm1
  LN: blocks.8.norm2
  LN: blocks.9.norm1
  LN: blocks.9.norm2
  LN: blocks.10.norm1
  LN: blocks.10.norm2
  LN: blocks.11.norm1
  LN: blocks.11.norm2
  LN: norm
Replaced Softmax in 12 attention modules.
  Attn: blocks.0.attn
  Attn: blocks.1.attn
  Attn: blocks.2.attn
  Attn: blocks.3.attn
  Attn: blocks.4.attn
  Attn: blocks.5.attn
  Attn: blocks.6.attn
  Attn: blocks.7.attn
  Attn: blocks.8.attn
  Attn: blocks.9.attn
  Attn: blocks.10.attn
  Attn: blocks.11.attn

=== Joint full-LENS training ===
Trainable: 5,940,294 / 5,940,294 (100.0000%)

Joint full-model training: trainable parameters 5,940,294 / 5,

Epoch 001/50 | lr=7.493e-05 | train loss=132768.6860 top1=0.5490 | val loss=1.5401 top1=0.6243 top5=0.8495 | train time=23m 20.3s | val time=32.6s | max grad norm=2.045e+12


Epoch 002/50 | lr=7.470e-05 | train loss=1.6845 top1=0.6038 | val loss=1.4881 top1=0.6335 top5=0.8557 | train time=23m 21.4s | val time=32.0s | max grad norm=1.180e+02


Epoch 003/50 | lr=7.434e-05 | train loss=1.6328 top1=0.6137 | val loss=1.4534 top1=0.6423 top5=0.8599 | train time=23m 20.0s | val time=32.4s | max grad norm=1.085e+02


Epoch 004/50 | lr=7.382e-05 | train loss=1.5958 top1=0.6210 | val loss=1.4496 top1=0.6444 top5=0.8623 | train time=23m 27.8s | val time=32.5s | max grad norm=1.248e+02


Epoch 005/50 | lr=7.316e-05 | train loss=1.5747 top1=0.6248 | val loss=1.4232 top1=0.6481 top5=0.8649 | train time=23m 19.7s | val time=31.9s | max grad norm=5.690e+01


... (42 outputs left out)

Epoch 048/50 | lr=2.957e-07 | train loss=0.9460 top1=0.7621 | val loss=1.2469 top1=0.7083 top5=0.8994 | train time=14m 59.6s | val time=31.6s | max grad norm=2.504e+01


Epoch 049/50 | lr=7.400e-08 | train loss=0.9432 top1=0.7625 | val loss=1.2480 top1=0.7080 top5=0.8996 | train time=14m 58.9s | val time=31.6s | max grad norm=2.476e+01


Epoch 050/50 | lr=0.000e+00 | train loss=0.9412 top1=0.7635 | val loss=1.2478 top1=0.7082 top5=0.8995 | train time=14m 57.9s | val time=31.7s | max grad norm=3.057e+01
DONE | best validation top1=0.7092 | last completed epoch=50
Loaded checkpoint: runs/lens_deit_tiny_joint_mlp_base2/best.pt | epoch=47 | val_top1=0.70924 | current device=cuda:0



[Best full-LENS DeiT-Tiny]
Validation: loss=1.2469, top1=0.7092, top5=0.8991, time=31.7s

=== Summary ===
Standard pretrained top1: 0.7221 | top5: 0.9110
Full LENS top1:           0.7092 | top5: 0.8991
Top-1 delta vs standard:  -1.282 pp
Best checkpoint: /home/yuhao/project/LENS/DeiT/Base2/seed1024/runs/lens_deit_tiny_joint_mlp_base2/best.pt
Resume checkpoint: /home/yuhao/project/LENS/DeiT/Base2/seed1024/runs/lens_deit_tiny_joint_mlp_base2/last.pt
Epoch text log: /home/yuhao/project/LENS/DeiT/Base2/seed1024/runs/lens_deit_tiny_joint_mlp_base2/training_log.txt


## Notes

- Model: plain `deit_tiny_patch16_224.fb_in1k` (not distilled), 197 tokens.
- MLPSoftmax is direct row-wise `197 -> 32 -> 1`; no recurrent loop remains.
- This notebook uses **Base-2** MLPSoftmax.
- LearnedNorm `b`/beta is intentionally **unclamped**.
- `AUTO_RESUME=True` resumes after the most recently completed epoch. If the
  process stops inside an epoch, that epoch is repeated.
- All checkpoint loads go to CPU first, then to the selected CUDA device.
- E and Base-2 runs use separate output directories and metadata validation,
  so accidentally loading the wrong variant is rejected.
- Watch progress from a shell with:
  `tail -f <run-dir>/training_log.txt`
- To deliberately start a fresh run, use a new `JOINT_OUT_DIR` or remove the
  previous run directory.
